# Домашнее задание: Реализация RAG-системы с поиском по собственной базе документов (ChromaDB)

## Описание
В данном ноутбуке мы:
1. Настроим окружение и установим необходимые библиотеки.
2. Подготовим синтетический датасет.
3. Создадим эмбеддинги с использованием `sentence-transformers`.
4. Настроим и проиндексируем данные в ChromaDB (изучим HNSW).
5. Реализуем семантический поиск, фильтрацию по метаданным.
6. Проведем анализ производительности.

In [ ]:
import sys
import os
import time
import pandas as pd
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
from typing import List, Dict
from datasets import load_dataset

print(f"Python executable: {sys.executable}")
print(f"ChromaDB version: {chromadb.__version__}")
# Убедимся, что мы в виртуальном окружении

In [ ]:


print("Загрузка датасета IlyaGusev/gazeta...")
dataset = load_dataset("IlyaGusev/gazeta", split="test[:5000]")

documents_data = []

print("Обработка данных...")
for i, item in enumerate(dataset):
    
    full_text = f"{item['title']}. {item['summary']}"
    
    
    category = "general"
    if "спорт" in full_text.lower():
        category = "sport"
    elif "технолог" in full_text.lower() or "интернет" in full_text.lower() or "наука" in full_text.lower():
        category = "tech"
    elif "политика" in full_text.lower() or "путин" in full_text.lower() or "закон" in full_text.lower():
        category = "politics"
    elif "экономика" in full_text.lower() or "рубль" in full_text.lower() or "банк" in full_text.lower():
        category = "economy"
        
    documents_data.append({
        "id": str(i),
        "text": full_text,
        "category": category,
        "source_url": item.get('url', ''),
        "date": item.get('date', '')
    })

df = pd.DataFrame(documents_data)
print(f"Загружено документов: {len(df)}")
print("Пример данных:")
display(df.head())

In [ ]:


model_name = 'intfloat/multilingual-e5-base'
print(f"Загрузка модели {model_name}...")
embedding_model = SentenceTransformer(model_name)

sample_text = "passage: Пример текста"
sample_embedding = embedding_model.encode(sample_text)
print(f"Модель загружена. Размерность вектора: {len(sample_embedding)}")

# Часть 1. Настройка и индексация (ChromaDB)

In [ ]:

client = chromadb.Client()

try:
    client.delete_collection("my_rag_collection")
except Exception:

    pass

collection = client.create_collection(
    name="my_rag_collection",
    metadata={
        "hnsw:space": "cosine", 
        "hnsw:construction_ef": 128,
        "hnsw:M": 24 
    }
)

print("Коллекция создана.")

In [ ]:
# 5. Индексация документов и добавление метаданных

# Для E5 модели нужно добавить префикс "passage: "
texts_for_embedding = [f"passage: {text}" for text in df['text']]

# Генерация эмбеддингов
# Увеличим batch_size для ускорения, если есть GPU (но на CPU тоже чуть поможет с overhead)
start_emb = time.time()
embeddings = embedding_model.encode(texts_for_embedding, batch_size=32, show_progress_bar=True)
print(f"Эмбеддинги сгенерированы за {time.time() - start_emb:.2f} сек.")

# Подготовка данных для добавления
ids = df['id'].tolist()
documents = df['text'].tolist() # В базу кладем чистый текст без префикса (чтобы читать человеком)
metadatas = df[['category', 'source_url', 'date']].to_dict(orient='records')

# ChromaDB ограничивает размер батча (обычно ~40k, у нас 1k, так что все ок)
print("Добавление в коллекцию ChromaDB...")
collection.add(
    embeddings=embeddings,
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Индексация завершена. Всего документов в коллекции: {collection.count()}")

# Часть 2. Реализация поиска


In [ ]:


def search(query_text: str, n_results: int = 3, category_filter: str = None):

    query_prepared = f"query: {query_text}"
    query_embedding = embedding_model.encode([query_prepared])

    where_filter = None
    if category_filter:
        where_filter = {"category": category_filter}

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        where=where_filter,
        include=['documents', 'metadatas', 'distances']
    )
    
    return results

print("--- Запрос: 'влияние ключевой ставки' (ожидаем экономику) ---")
res1 = search("влияние ключевой ставки ЦБ", n_results=3)
for i in range(len(res1['ids'][0])):
    print(f"Doc ID: {res1['ids'][0][i]}")
    print(f"Dist: {res1['distances'][0][i]:.4f} | Cat: {res1['metadatas'][0][i]['category']}")
    print(f"Text snippet: {res1['documents'][0][i][:100]}...\n")
    
print("--- Запрос: 'новые смартфоны и гаджеты' (Фильтр: category='tech') ---")
res2 = search("новые смартфоны и гаджеты", n_results=3, category_filter='tech')
for i in range(len(res2['ids'][0])):
    if not res2['ids'][0]:
        print("Ничего не найдено с таким фильтром.")
        break
    print(f"Doc ID: {res2['ids'][0][i]}")
    print(f"Dist: {res2['distances'][0][i]:.4f} | Cat: {res2['metadatas'][0][i]['category']}")
    print(f"Text snippet: {res2['documents'][0][i][:100]}...\n")

In [ ]:
def benchmark_search(n_queries=100, top_k=5):
    start_time = time.time()

    base_query = "анализ данных и алгоритмы"

    query_vec = embedding_model.encode([base_query])
    
    for _ in range(n_queries):
        collection.query(
            query_embeddings=query_vec,
            n_results=top_k
        )
        
    end_time = time.time()
    avg_time = (end_time - start_time) / n_queries
    print(f"Среднее время поиска (на {n_queries} запросов, top-k={top_k}): {avg_time*1000:.4f} ms")

print("Тест производительности:")
benchmark_search(n_queries=100, top_k=1)
benchmark_search(n_queries=100, top_k=5)
benchmark_search(n_queries=100, top_k=10)

In [ ]:
queries = [
    "зеленые технологии",
    "история компьютеров",
    "разнообразие животного мира",
    "защита информации"
]

batch_embeddings = embedding_model.encode(queries)

start_batch = time.time()
batch_results = collection.query(
    query_embeddings=batch_embeddings,
    n_results=2
)
end_batch = time.time()

print(f"Время выполнения батча из {len(queries)} запросов: {(end_batch - start_batch)*1000:.4f} ms")

for i, q in enumerate(queries):
    print(f"\nQuery: {q}")
    for j in range(len(batch_results['ids'][i])):
        print(f" - Found: {batch_results['documents'][i][j]} (Dist: {batch_results['distances'][i][j]:.4f})")